# Phase 1 — références zero-shot

Ce notebook mesure ce que donnent les modèles publics **tels quels**, sans
aucun entraînement, sur chaque maillon de la chaîne :

| Maillon | Modèle | Corpus | Métriques |
|---|---|---|---|
| ASR | `openai/whisper-small` | Jeli-ASR | WER/CER strict et relâché |
| ASR | `facebook/mms-1b-all` (adaptateur `bam`) | Jeli-ASR | WER/CER + propagation vers la MT |
| MT | `facebook/nllb-200-distilled-600M` | Bayelemabaga (écrit) et Jeli-ASR (oral) | chrF++, BLEU, chrF++ replié vers le bambara |
| TTS | `facebook/mms-tts-bam` | phrases de Jeli-ASR | échantillon audio + grille MOS |

Ce sont les « références basses » du mémoire : tout fine-tuning (phase 3)
se juge par rapport à elles. Pas de variante bout-en-bout ici, car la tâche
`translate` de Whisper ne produit que de l'anglais. Elle n'existe qu'après
fine-tuning.

**Avant de lancer** : *Exécution → Modifier le type d'exécution → GPU (T4)*.
Compter environ 30 minutes pour l'ensemble.

## 1. Installation

In [ ]:
# Branche de travail ; remplacer par "main" une fois la branche fusionnée.
BRANCH = "claude/chat-ia-bambara-dho8oa"

!git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd projet
!pip install -q transformers datasets sentencepiece jiwer sacrebleu soundfile librosa

## 2. Où ranger les résultats

Le disque de Colab est effacé à la fin de la session : on écrit donc les
rapports sur Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/bambara-phase1"
!mkdir -p "$OUT"

## 3. Accès aux jeux de données

Si un jeu est « gated » sur Hugging Face, accepter ses conditions sur sa
page, puis se connecter ici avec un jeton en lecture
(<https://huggingface.co/settings/tokens>). Sinon, passer cette cellule.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Vérifier les colonnes

Les noms de colonnes de ces jeux ne sont pas stables d'une version à
l'autre. Le code les détecte ; cette cellule montre ce qu'il a trouvé.

Si Jeli-ASR demande un sous-ensemble, le message d'erreur liste les noms
possibles : le renseigner dans `JELI_CONFIG` et relancer la cellule.
Si une colonne est mal détectée, ajouter `--bm-column ...` ou
`--fr-column ...` aux commandes suivantes.

In [ ]:
JELI_CONFIG = ""  # ex. "bam-asr-oza" si le jeu en demande un

from eval.corpora import JELI_ASR, BAYELEMABAGA, load_split, detect_columns

jeli = load_split(JELI_ASR, config=JELI_CONFIG or None)
print(jeli)
print("Jeli-ASR    :", detect_columns(jeli.features, need_audio=True).describe())

bayel = load_split(BAYELEMABAGA)
print(bayel)
print("Bayelemabaga:", detect_columns(bayel.features, need_fr=True).describe())

JELI = f"--dataset {JELI_ASR}" + (f" --config {JELI_CONFIG}" if JELI_CONFIG else "")

## 5. ASR

300 énoncés tirés au hasard (graine 0) dans la partition de test. Whisper
reçoit le token de langue swahili, le même que pour le fine-tuning : entre
la référence zero-shot et le modèle affiné, seuls les poids changent.

MMS est la référence sérieuse : il a vu du bambara à l'entraînement.
`--with-mt` traduit aussi sa sortie avec NLLB, pour mesurer dès maintenant
ce que les erreurs d'ASR coûtent à la traduction (propagation d'erreurs).

In [ ]:
!python -m eval.baselines --out "$OUT" asr $JELI --model openai/whisper-small --limit 300

In [ ]:
!python -m eval.baselines --out "$OUT" asr $JELI --model facebook/mms-1b-all \
    --kind ctc --target-lang bam --with-mt --limit 300

## 6. Traduction

Deux corpus, parce qu'ils ne mesurent pas la même chose : Bayelemabaga est
de l'écrit (presse, livres, textes religieux), Jeli-ASR de l'oral transcrit,
plus proche de ce que l'assistant entendra.

Vers le bambara, le rapport donne aussi un chrF++ sur la forme repliée
(ɛ→e, ɔ→o...). L'écart avec le chrF++ strict mesure ce que coûte la seule
graphie.

In [ ]:
!python -m eval.baselines --out "$OUT" mt --dataset RobotsMaliAI/bayelemabaga --limit 500

In [ ]:
!python -m eval.baselines --out "$OUT" mt $JELI --limit 500

## 7. Synthèse vocale

Aucune métrique automatique ne juge correctement la prosodie d'une langue à
tons non notés. On produit donc 20 phrases et une grille `mos.csv` à faire
remplir par au moins trois auditeurs natifs (note de 1 à 5 par phrase).

In [ ]:
!python -m eval.baselines --out "$OUT" tts $JELI --limit 20

In [ ]:
# Écouter quelques phrases.
import csv, glob
from IPython.display import Audio, display

dossier = sorted(glob.glob(f"{OUT}/tts-*"))[-1]
with open(f"{dossier}/mos.csv", encoding="utf-8") as fh:
    for row in list(csv.DictReader(fh))[:3]:
        print(row["texte_bm"])
        display(Audio(f"{dossier}/{row['fichier']}"))

## 8. Tableau récapitulatif — à reprendre dans le mémoire

In [ ]:
!python -m eval.compare "$OUT"/zero-shot-*.json

## Lire les résultats

- **WER strict contre relâché** : l'écart est la part d'erreur due à la
  seule orthographe. Un gros écart signifie que le modèle entend bien mais
  écrit en ASCII.
- **Propagation d'erreurs** : la perte de chrF++ entre la traduction de la
  transcription de référence et celle de la sortie de l'ASR. C'est le coût
  de l'ASR pour la suite de la chaîne, celui que la variante bout-en-bout
  cherche à supprimer.
- **Latence et RTF** : mesurés sur le GPU de Colab, donc **pas**
  représentatifs du déploiement sur CPU (phase 5).
- **Intervalles de confiance** : sur 300 énoncés, quelques points de WER
  d'écart ne sont pas significatifs.

Chaque rapport (`.json`) consigne le modèle, le corpus, la partition, la
graine, l'appareil et les versions. Les fichiers `.details.jsonl` gardent
les sorties ligne à ligne pour l'analyse d'erreurs.